# Importations

In [1]:

import matplotlib.pyplot as plt
import sys
from IPython.core import ultratb
import dill
import numpy as np
from functions_synthetic_data_generation import *
from functions_synthetic_data_analysis import *
import matplotlib.animation as animation

sys.excepthook = ultratb.FormattedTB(call_pdb=False)

plt.style.use('../paper.mplstyle')

print(plt.get_backend())
%matplotlib qt5
print(plt.get_backend())

module://matplotlib_inline.backend_inline
qt5agg


# Parameters

In [ ]:
# simulations_folder_path = '/home/david/Documents/code/DDM_vXbis_synthetic_data_identical_drift'
simulations_folder_path = f'/home/david/Documents/code/DDM_vXbis_synthetic_data_identical_drift_0025'
simulations_folder_path = '/home/david/Documents/code/loop_drift_estimation_output/drift_2/'
simulations_folder_path = '/home/david/Documents/code/loop_test_drift_estimation_output/drift_2/'

# simulations_folder_path = f'/home/tom/Code/ddm_simulations/simulations_batches/'

n_simulations = 50
batch_index = 0
simulations_indexes = np.arange(n_simulations)

# with open(f'{simulations_folder_path}/n_{n_simulations}/simulations_batch_{n_simulations}_test_{batch_index+1}.pkl', 'rb') as file:
#     synthetic_data = dill.load(file)

with open(f'{simulations_folder_path}/simulation_set_{batch_index}.pkl', 'rb') as file:
    synthetic_data = dill.load(file)


test_data = [synth_data['choices'] for synth_data in synthetic_data]

# Plots

## All simulations

In [10]:
fig=plt.figure(figsize=(4, 2), dpi=300, constrained_layout=False, facecolor='w')
gs = fig.add_gridspec(1, 1, hspace=0.5,)
row = gs[0,0].subgridspec(2, 1)


for i, simulation_index in enumerate(simulations_indexes):

    ddm_result = synthetic_data[simulation_index]

    choice_sequence = ddm_result['choices']
    p_cw_sequence = ddm_result['p_cw']
    reward_sequence = ddm_result['rewards']

    steps = np.arange(len(choice_sequence))

    colors = ['violet' if i==1 else 'purple' for i in reward_sequence]

    ax1 = plt.subplot(row[0,0])
    ax1.scatter(steps, choice_sequence, label='Action Sequence', color=colors, marker='+')
    ax1.set_ylabel('Action')
    ax1.set_xticks([])
    ax1.set_yticks([0,1])
    ax1.set_yticklabels(['CCW','CW'])

    ax2 = plt.subplot(row[1,0])
    ax2.plot(steps, p_cw_sequence, label='Probability Sequence', alpha=0.2)
    
    ax2.set_xlabel('Step')
    ax2.set_ylabel('Probability\nto do CW')
    ax2.set_xticks(steps[::5])
    ax2.set_ylim([-0.05,1.05])

plt.show()

## One simulation

In [8]:
fig=plt.figure(figsize=(4, 3), dpi=300, constrained_layout=False, facecolor='w')
gs = fig.add_gridspec(1, 1, hspace=0.5,)
row = gs[0,0].subgridspec(2, 1)

simulation_index = 4

ddm_result = synthetic_data[simulation_index]

choice_sequence = ddm_result['choices']
p_cw_sequence = ddm_result['p_cw']
reward_sequence = ddm_result['rewards']

steps = np.arange(len(choice_sequence))

colors = ['violet' if i==1 else 'indigo' for i in reward_sequence]

ax1 = plt.subplot(row[0,0])
ax1.scatter(steps, choice_sequence, label='Action Sequence', color=colors, marker='+')
ax1.set_ylabel('Action')
ax1.set_xticks([])
ax1.set_yticks([0,1])
ax1.set_yticklabels(['CCW','CW'])

ax2 = plt.subplot(row[1,0])
ax2.plot(steps, p_cw_sequence, label='Probability Sequence', alpha=1, marker='+')

ax2.set_xlabel('Step')
ax2.set_ylabel('Probability\nto do CW')
ax2.set_xticks(steps[::5])
ax2.set_ylim([-0.05,1.05])

plt.show()

## Probability distribution

In [4]:
p_cw_init_range = np.linspace(0,1,100)

temp_args_dict = synthetic_data[0]["parameters"]
batch_size = 100
temp_synthetic_data = run_simulations_batch_random_init(temp_args_dict, batch_size, p_cw_init_range)
stacked_proba_sequences_list = []
        
for i in range(batch_size):

    ddm_result = temp_synthetic_data[i]

    choice_sequence = ddm_result['choices']
    p_cw_sequence = ddm_result['p_cw']
    reward_sequence = ddm_result['rewards']

    stacked_proba_sequences_list.append(p_cw_sequence)

proba_distributions = np.transpose(stacked_proba_sequences_list)


100%|██████████| 100/100 [00:00<00:00, 351.98it/s]


In [8]:
temp_args_dict


{'p_cw_reward': 0.8,
 'p_ccw_reward': 0,
 'p_cw_init': np.float64(0.0),
 'steps_number': 40,
 'noise_amplitude': 0.1,
 'drift_matrix': array([[ 0.05, -0.05],
        [ 0.  ,  0.  ]]),
 'drift_init': 0.0}

In [ ]:
# number_of_frames = 40
# data = proba_distributions #np.random.rand(n, number_of_frames)

# def update_hist(num, data):
#     plt.cla()
#     plt.hist(data[num], bins=np.linspace(0,1,20))
#     plt.xlim([0,1])
#     plt.ylim([0,len(data[num])])


# fig = plt.figure()
# hist = plt.hist(data[0])
# animation_ = animation.FuncAnimation(fig, update_hist, number_of_frames, fargs=(data, ) )
# plt.show()


In [6]:
fig=plt.figure(figsize=(8, 3), dpi=300, constrained_layout=False, facecolor='w')
gs = fig.add_gridspec(1, 1, wspace=0.5,)
row = gs[0,0].subgridspec(1, 8,wspace=0.5)


for n,i in enumerate(range(0,40,5)):

    ax = plt.subplot(row[0,n])
    ax.hist(proba_distributions[i], bins=np.linspace(0,1,20))
    ax.set_xlim([0,1])
    ax.set_ylim([0,50])

    ax.set_xlabel(f'P_{i}(CW)')
    
    if n==0:

        ax.set_ylabel('Counts')
    